# COCO Computer Vision Detection: Tutorial ฉบับมือใหม่

ยินดีต้อนรับสู่คู่มือการทำ **Computer Vision Object Detection** ด้วยชุดข้อมูลมาตรฐานโลก **COCO (Common Objects in Context)**!

---

### โครงสร้างไดเรกทอรีมาตรฐานในโปรเจกต์:
- **Pretrained Weights**: `/workspace/artifacts/pretrained`
- **Datasets**: `/workspace/data` (เช่น `coco128/`, `coco8/`)
- **Training Runs & Metrics**: `/workspace/artifacts/runs`
- **Prediction Outputs**: `/workspace/artifacts/predictions`
- **Tutorial Code & Samples**: `/workspace/cv-coco-detection` (`bus.jpg`, `zidane.jpg`)


## 1. ตรวจสอบสภาพแวดล้อม (GPU & PyTorch & Ultralytics)
ตรวจสอบสถานะ GPU และ Library ในเครื่อง

In [1]:
import os
import torch
import ultralytics
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics Version: {ultralytics.__version__}")


PyTorch Version: 2.14.0a0+4fdf77b940.nv26.08
CUDA Available: True
GPU Device: NVIDIA GB10
Ultralytics Version: 8.4.183


## 2. Quickstart: ตรวจจับวัตถุด้วย Pre-trained Model
โหลดโมเดลจาก `/workspace/artifacts/pretrained/yolo11n.pt` และบันทึกผลลัพธ์ลงใน `/workspace/artifacts/predictions`

In [ ]:
PRETRAINED_MODEL = "/workspace/artifacts/pretrained/yolo11n.pt"
PREDICTIONS_DIR = "/workspace/artifacts/predictions"

# โหลดโมเดล YOLO11 Nano จากโฟลเดอร์ artifacts/pretrained
model = YOLO(PRETRAINED_MODEL)

# รันการตรวจจับภาพ (ดึงภาพตัวอย่าง bus.jpg)
sample_img = "bus.jpg" if os.path.exists("bus.jpg") else "https://ultralytics.com/images/bus.jpg"

results = model.predict(
    source=sample_img,
    conf=0.5,
    save=True,
    project=PREDICTIONS_DIR,
    name="quickstart_demo",
    exist_ok=True
)

for idx, box in enumerate(results[0].boxes, 1):
    cls_name = model.names[int(box.cls[0])]
    conf = float(box.conf[0]) * 100
    print(f"[{idx}] ตรวจพบ: {cls_name} ({conf:.1f}%)")


## 3. ทำความรู้จัก COCO Dataset ใน `/workspace/data`
COCO มี 80 คลาสครอบคลุมวัตถุในชีวิตประจำวัน โดยข้อมูลจะถูกจัดเก็บไว้ใน `/workspace/data/coco128`

In [ ]:
from ultralytics.data.utils import check_det_dataset

# ตรวจสอบและดึงข้อมูลจาก coco128.yaml ใน /workspace/data
data_dict = check_det_dataset("coco128.yaml")
classes = data_dict["names"]

print(f"ตำแหน่งข้อมูล: {data_dict['path']}")
print(f"จำนวนคลาสทั้งหมด: {len(classes)} คลาส")
print("ตัวอย่างคลาส 10 อันดับแรก:")
for i in range(10):
    print(f"  Class {i}: {classes[i]}")


## 4. ฝึกสอนโมเดล (Training with Transfer Learning)
ฝึกต่อบนชุดข้อมูล COCO128 โดยเก็บผลการเทรนไว้ที่ `/workspace/artifacts/runs`

In [ ]:
RUNS_DIR = "/workspace/artifacts/runs"

train_results = model.train(
    data="coco128.yaml",
    epochs=3,
    imgsz=640,
    batch=16,
    project=RUNS_DIR,
    name="coco_train_demo",
    exist_ok=True
)

print("เทรนสำเร็จ! ดูน้ำหนักโมเดลได้ที่:", train_results.save_dir)


## 5. การประเมินผลความแม่นยำ (Validation & Metrics)
วัดประสิทธิภาพของโมเดลบนชุดข้อมูล COCO128

In [ ]:
metrics = model.val(data="coco128.yaml", project=RUNS_DIR, name="val")

print(f"mAP50:    {metrics.box.map50 * 100:.2f}%")
print(f"mAP50-95: {metrics.box.map * 100:.2f}%")


## 6. นำโมเดลไปทดสอบกับภาพใหม่ (Inference)
ทดสอบตรวจจับภาพ และเก็บผลลัพธ์ใน `/workspace/artifacts/predictions/evaluation_demo`

In [ ]:
test_img = "zidane.jpg" if os.path.exists("zidane.jpg") else "https://ultralytics.com/images/zidane.jpg"

predict_results = model.predict(
    source=test_img,
    conf=0.4,
    save=True,
    project=PREDICTIONS_DIR,
    name="evaluation_demo",
    exist_ok=True
)
print("บันทึกผลลัพธ์ภาพที่ตรวจจับเรียบร้อยแล้วใน:", os.path.join(PREDICTIONS_DIR, "evaluation_demo"))
